In [1]:
import pandas as pd
import numpy as np
from scipy.signal import savgol_filter, find_peaks, peak_widths
import matplotlib.pyplot as plt
import json
# import simplekml
import math
import matplotlib.dates as mdates
import ee
# ee.Authenticate()
ee.Initialize(project="ee-joshisur231")
import geemap
Map = geemap.Map()
import plotly.graph_objects as go
import ast
import geopandas as gpd

In [2]:
df = pd.read_csv(r"outputs\stable_nonCrop_points_with_ndvi2000-2022_frtcOnly.csv")
df["date"] = pd.to_datetime(df["time"], unit= "ms")
df = df.drop(columns=["time"])
df["point_id"] = df["system:index"].str.split("_").str[-1]
df = df.drop(columns=["system:index"])

In [3]:
stable_non_crop_ids = []
discarded_ids = []
ndvi_dict = {
    "point_id": [],
    "mean": [],
    "min": [],
    "max": [],
    "std": [],
    "status": [],      
    "lc2022": [],
    "total_peaks":[],
    "geo": []          
}

grouped_points = df.sort_values(by="point_id").groupby("point_id")

for point_id, group in grouped_points:
    ts_raw = group.set_index('date')['NDVI'].resample("16D").mean()

    total_valid_obs = ts_raw.count()

    if total_valid_obs < 200:
        # print(f"Insufficient Samples for {point_id}: {total_valid_obs}")
        discarded_ids.append(point_id)

        ndvi_dict["point_id"].append(point_id)
        ndvi_dict["lc2022"].append(group.iloc[0]["lc2022"])
        ndvi_dict["mean"].append(np.nan)
        ndvi_dict["min"].append(np.nan)
        ndvi_dict["max"].append(np.nan)
        ndvi_dict["std"].append(np.nan)
        ndvi_dict["status"].append("DISCARDED (Insufficient Samples)")
        ndvi_dict["total_peaks"].append(np.nan)
        ndvi_dict["geo"].append(group.iloc[0][".geo"])
        continue

    #Interpolate values across the entire 23 years
    ts_interp = ts_raw.interpolate(method="linear").bfill().ffill()

    # half_window = math.floor(year_valid_obs / 4) #timesat logic
    window = 11 #(2 * half_window) + 1
    
    smoothed_ndvi = savgol_filter(ts_interp.values, window_length = window, polyorder=2)
    smoothed_ndvi_series = pd.Series(smoothed_ndvi, index=ts_interp.index)

    peaks, properties = find_peaks(
        smoothed_ndvi, 
        height=0.35,         # Kept the same: reliably identifies active vegetation
        distance= 5,          # Lowered from 7: Allows peaks to be 80 days apart (Captures tight multi-cropping)
        width=(5, 15),       # Lowered from 7: 7 samples (80 days) captures the rapid Terai cycles, 15 caps the forests
        prominence=0.10      # Lowered from 0.12: Accounts for overlapping crop cycles where NDVI doesn't hit bare soil
    )
   
    peak_dates = ts_interp.index[peaks]

    yearly_classification = {}
    yearly_peaks = []
    
    overall_mean = smoothed_ndvi_series.mean()
    overall_min =  smoothed_ndvi_series.min()
    overall_max = smoothed_ndvi_series.max()
    overall_amplitude = overall_max - overall_min
    
    if overall_amplitude < 0.55 and overall_min > 0.25:
        stable_non_crop_ids.append(point_id)
        status_reason = "STABLE NON-CROP (Rangeland)"
            
        # 2. FOREST (Deciduous / Deep Canopy)
        # Rule: Massive canopy peak AND doesn't drop to bare dirt
    elif overall_max > 0.80 and overall_min > 0.25:
        stable_non_crop_ids.append(point_id)
        status_reason = "STABLE NON-CROP (Forest)"
        
    # 3. BARREN / WATER / SNOW (Low Productivity)
    # Rule: Never gets green enough to be a crop
    elif overall_max < 0.45:
        stable_non_crop_ids.append(point_id)
        status_reason = "STABLE NON-CROP (Barren/Water)"
        
    # 4. CROP OR DISTURBANCE (The Harvest Signature)
    # Rule: Massive swing AND drops to bare dirt (Min <= 0.25)
    else:
        discarded_ids.append(point_id)
        status_reason = "DISCARDED (Crop or Disturbance)"
            

    ndvi_dict["point_id"].append(point_id)
    ndvi_dict["lc2022"].append(group.iloc[0]["lc2022"])
    ndvi_dict["mean"].append(round(overall_mean, 4))
    ndvi_dict["min"].append(round(overall_min, 4))
    ndvi_dict["max"].append(round(overall_max, 4))
    ndvi_dict["std"].append(round(np.std(smoothed_ndvi), 4))
    ndvi_dict["total_peaks"].append(len(peaks))
    ndvi_dict["status"].append(status_reason)
    ndvi_dict["geo"].append(group.iloc[0][".geo"])
  

    # fig = go.Figure()
    # # 2. Raw Resampled (Scatter Dots)
    # fig.add_trace(go.Scatter(
    #     x=ts_raw.index, 
    #     y=ts_raw.values,
    #     mode='markers',
    #     name='Raw Resampled',
    #     marker=dict(color='grey', opacity=0.5),
    #     showlegend=True
    # ))

    # # 3. Raw Resampled (Dashed Connecting Line)
    # fig.add_trace(go.Scatter(
    #     x=ts_raw.index, 
    #     y=ts_raw.values,
    #     mode='lines',
    #     name='Raw Resampled (Line)',
    #     line=dict(color='black', width=1, dash='dash'),
    #     opacity=0.6,
    #     showlegend=False # Matplotlib showed 1 legend item for both, this keeps it clean
    # ))

    # # 4. Savgol Smoothed (Solid Green Line)
    # fig.add_trace(go.Scatter(
    #     x=ts_interp.index, 
    #     y=smoothed_ndvi,
    #     mode='lines',
    #     name='Savgol Smoothed',
    #     line=dict(color='green', width=1.5),
    #     opacity=0.7
    # ))

    # # 5. Detected Peaks (Red X Markers)
    # if len(peaks) > 0:
    #     fig.add_trace(go.Scatter(
    #         x=ts_interp.index[peaks], 
    #         y=smoothed_ndvi[peaks],
    #         mode='markers',
    #         name='Detected Peaks',
    #         marker=dict(symbol='x', color='red', size=10, line=dict(width=2, color='red'))
    #     ))


    # info_text = (
    #     f"Mean, Min, Max: {round(overall_mean, 2)}, {round(overall_min, 2)}, {round(overall_max, 2)}<br>"
    #     f"Total Peaks (2000-2022): {len(peaks)}"
    #     f"LC 2022: {group.iloc[0]["lc2022"]}"
    # )

    # fig.add_annotation(
    #     x=0.7,
    #     y=1.4,
    #     xref="paper",
    #     yref="paper",
    #     text=info_text,
    #     showarrow=False,
    #     align="left",
    #     bgcolor="rgba(255, 255, 255, 0.8)", # White background with 80% opacity
    #     bordercolor="gray",
    #     borderwidth=1,
    #     borderpad=6,
    #     font=dict(size=11),
    #     xanchor="left",
    #     yanchor="top"
    # )

    # # 7. Layout formatting (Mimicking Matplotlib styling)
    # fig.update_layout(
    #     title=f"Continuous 23-Year Phenology Profile",
    #     height=450,           # Similar to figsize=(..., 5)
    #     plot_bgcolor='white', # Removes Plotly's default gray background
        
    #     # Y-Axis Settings (-0.1 to 1.0)
    #     yaxis=dict(
    #         range=[-0.1, 1.0], 
    #         gridcolor='rgba(128,128,128,0.2)', # Faint grid lines
    #         zerolinecolor='rgba(128,128,128,0.5)'
    #     ),
        
    #     # X-Axis Settings (1-year ticks, 45 degree angle)
    #     xaxis=dict(
    #         dtick="M12",          # Major ticks every 12 Months
    #         tickformat="%Y",      # Show only the Year
    #         tickangle=45,
    #         gridcolor='rgba(128,128,128,0.2)'
    #     ),
        
    #     # Legend position (Lower Right)
    #     legend=dict(
    #         x=0.99,
    #         y=0.02,
    #         xanchor="right",
    #         yanchor="bottom",
    #         bgcolor="rgba(255,255,255,0.7)"
    #     )
    # )

    # fig.show()
    
print(f"Total points analyzed: {len(grouped_points)}")
print(f"Pure Non Cropland Points Kept: {len(stable_non_crop_ids)}")
print(f"Points Discarded: {len(discarded_ids)}")

results_df = pd.DataFrame(ndvi_dict)

Total points analyzed: 9272
Pure Non Cropland Points Kept: 3673
Points Discarded: 5599


In [20]:
results_df.to_csv("outputs\phenology_verified_samples\stable_nonCrop_phenology_classification_results.csv", index=False)

In [12]:
results_df["lc2022"].value_counts()

lc2022
4     6230
9     1071
2      800
10     635
11     328
5       90
1       71
6       46
8        1
Name: count, dtype: int64

In [4]:
stable_categories = [
    'STABLE NON-CROP (Rangeland)', 
    'STABLE NON-CROP (Forest)', 
    "STABLE NON-CROP (Barren/Water)"
]
non_stable_categories =[
    'DISCARDED (Insufficient Samples)',
    'DISCARDED (Crop or Disturbance)'
]
stable_nonCrop_df = results_df[results_df["status"].isin(stable_categories)]
non_stable_nonCrop_df = results_df[results_df["status"].isin(non_stable_categories)]

In [14]:
stable_nonCrop_df["lc2022"].value_counts().sort_index()

lc2022
1        7
4     3123
5       33
6       23
9      288
10      61
11     138
Name: count, dtype: int64

In [18]:
non_stable_nonCrop_df[(non_stable_nonCrop_df["status"]=='DISCARDED (Crop or Disturbance)') & (non_stable_nonCrop_df["lc2022"]==10)].to_csv(r"outputs\test\discarded_rangeland.csv")

In [20]:
for status in list(results_df["status"].unique()):
    print(r"output/test/"+status.replace("/", "_")+".csv")
    results_dis = results_df[results_df["status"] == status]
    results_dis["coords"] = results_dis["geo"].apply(lambda x: ast.literal_eval(x)["coordinates"])
    results_dis["x"] = results_dis["coords"].apply(lambda x: x[0])
    results_dis["y"] = results_dis["coords"].apply(lambda x: x[1])
    results_dis.to_csv(r"outputs/test/" + status.replace("/", "_")+ ".csv")


output/test/STABLE NON-CROP (Rangeland).csv
output/test/STABLE NON-CROP (Forest).csv
output/test/DISCARDED (Insufficient Samples).csv
output/test/DISCARDED (Crop or Disturbance).csv
output/test/STABLE NON-CROP (Barren_Water).csv


In [ ]:
results_dis = results_df[results_df["status"] == "DISCARDED (Crop or Disturbance)"]

In [21]:
results_dis["lc2022"].value_counts()

lc2022
4     2021
9      506
10     236
11      94
5       37
6       33
1       11
Name: count, dtype: int64

## Prepare for manual validation

In [19]:
import pandas as pd
import ee
ee.Initialize(project = "ee-joshisur231")
import json
# import simplekml
import geopandas as gpd
import geemap
Map = geemap.Map()
import ast

In [20]:
stable_categories = [
    'STABLE NON-CROP (Rangeland)', 
    'STABLE NON-CROP (Forest)', 
    "STABLE NON-CROP (Barren/Water)"
]
geo_region = ee.Image("projects/ee-joshisur231/assets/pa_effectiveness/geoReg_nepal").rename("geoReg")
lc_2022 = ee.ImageCollection("projects/ee-joshisur231/assets/landcover_frtc_2000-2022_nepal").filter(ee.Filter.eq("system:index", "lc2022")).first()
strata = lc_2022.multiply(100).add(geo_region).rename("strata")
df = pd.read_csv(r"outputs\phenology_verified_samples\stable_nonCrop_phenology_classification_results.csv")
df = df[df["status"].isin(stable_categories)]

In [21]:
forest_subset = df[df['lc2022'] == 4].sample(n=500, random_state=45)
non_forest_data = df[df['lc2022'] != 4]
balanced_df = pd.concat([forest_subset, non_forest_data])

balanced_df["coords"] = balanced_df["geo"].apply(lambda x: ast.literal_eval(x)["coordinates"])
balanced_df["x"] = balanced_df["coords"].apply(lambda x: x[0])
balanced_df["y"] = balanced_df["coords"].apply(lambda x: x[1])

In [49]:
ee_final_stable_noncrop = geemap.df_to_ee(
    balanced_df,
    latitude = "y",
    longitude = "x" 
)

ee_final_stable_noncrop_with_strata = strata.reduceRegions(collection = ee_final_stable_noncrop, scale = 30, crs="EPSG:32645", reducer=ee.Reducer.first()).map(lambda feat: feat.set("strata", feat.get("first")))

final_stable_noncrop_withStrata = geemap.ee_to_df(ee_final_stable_noncrop_with_strata, remove_geom=True)

In [62]:
final_stable_noncrop_withStrata.groupby("strata").size()

strata
102       7
401     138
402     320
403      48
501      17
502       3
503      11
601       2
602      19
603       2
901     284
902       5
1001     34
1002     23
1003      6
1101     52
1102     75
1103      2
dtype: int64

In [68]:
sample_size = {
    102: 7,
    401: 17,
    402: 35,
    403: 22,
    501: 7,
    502: 2,
    503: 11,
    601: 2,
    602: 16,
    603: 2,
    901: 18,
    902: 2,
    1001: 11,
    1002: 5,
    1003: 4,
    1101: 9,
    1102: 9,
    1103: 2
}

def safe_sample(group):
    target_n = sample_size.get(group.name, 0)
    
    actual_n = min(target_n, len(group))
    return group.sample(n=actual_n)

final_stable_noncrop_withStrata = final_stable_noncrop_withStrata.dropna(subset="strata")
final_stable_noncrop_withStrata['strata'] = final_stable_noncrop_withStrata['strata'].astype(int)
stable_valid_samples = final_stable_noncrop_withStrata.groupby("strata", group_keys=False).apply(lambda x: x.sample(n=sample_size.get(x.name, 0)))

In [72]:
stable_valid_samples.to_csv(r"outputs\phenology_verified_for_validation\final_stable_nonCrop_manual_validation_samples.csv", index=False)

In [ ]:
# gdf = gpd.GeoDataFrame(stable_valid_samples[['geoReg', 'slope', 'orig_x','orig_y',
# 'mean', 'min','max',  'std',  'max_consecutive_non_crop', 'total_non_crop','total_peaks','status',
#   'point_id']], geometry= gpd.points_from_xy(stable_valid_samples.orig_x, stable_valid_samples.orig_y), crs="EPSG:4326")
# gdf.to_file(r"outputs/spatial_data/new/stable_crop_for_manual_validation_new.shp", driver="ESRI Shapefile")

In [79]:
df = pd.read_csv(r"outputs\phenology_verified_for_validation\final_stable_nonCrop_manual_validation_samples.csv")
gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.x, df.y), crs="EPSG:4326")

gdf = gdf.to_crs(epsg="32644")
gdf_buffer = gdf.buffer(15, cap_style=3)

In [80]:
gdf_buffer.to_file(r"outputs/spatial_data/new/stable_noncrop_for_manual_validation_grids_new.shp", driver="ESRI Shapefile")

## Validation Metrics

In [3]:
from helpers import config
import pandas as pd
from sklearn import metrics
import matplotlib.pyplot as plt
from sklearn import metrics

loaded config!


In [11]:
df = pd.read_csv(r"outputs\phenology_verified_for_validation\final_stable_nonCrop_manual_validation_samples.csv")

y_true = df["actual"]
y_pred = df["predicted"]
print("Final validation report")
conf_matrix = pd.crosstab(
    y_true, 
    y_pred, 
    rownames=['Actual'], 
    colnames=['Predicted']
)
print(conf_matrix)

print("\n=== CLASSIFICATION REPORT ===")
print(metrics.classification_report(
    y_true, 
    y_pred, 
    # target_names=['Discarded (Non-Crop)', 'Stable Cropland']
))

Final validation report
Predicted    1
Actual        
0            4
1          177

=== CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         4
           1       0.98      1.00      0.99       177

    accuracy                           0.98       181
   macro avg       0.49      0.50      0.49       181
weighted avg       0.96      0.98      0.97       181



e:\Software\conda_envs\gee\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
e:\Software\conda_envs\gee\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
e:\Software\conda_envs\gee\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [9]:
df[df["actual"] != 0]["lc"].value_counts().sum()

np.int64(177)